# Formation-window screen
Screens pairs using only data from `F_START` to `AS_OF`, with the settings in `code/screen_config.py`.

- **`MODE = "tuning"`**: screens every same-industry pair in the pool, ranks the eligible ones by CADF p-value, and keeps the top 30 as the tuning set (T001). No LLM, no baselines.
- **`MODE = "batch"`**: screens a skill batch (`PAIRS_STR`) plus random and same-industry baselines with the same trial count.

The last cell prints `screen.csv` and a `pair_registry.md` entry; lift them into the repo with `tools/extract_blocks.py`. Never edit `PAIRS_STR` after registering.

In [ ]:
from datetime import datetime, timedelta
import numpy as np, pandas as pd, random, json
from statsmodels.tsa.stattools import coint

# Metadata
MODE      = "tuning"                             # "tuning" = rule-based same-industry pool (no LLM, no baselines); "batch" = skill batch + baselines
BATCH_ID  = "T001"
SKILL_VER = "v1"                                 # batch mode only
CONFIG_VER = "v1"                                # picks the version in code/screen_config.py
AS_OF     = datetime(2016, 12, 31)

# Batch Mode (do not edit after registering)
PAIRS_STR = ""

def parse_pairs(text):
    """"A/B, C/D" -> [("A","B"), ("C","D")]; same rules as pairs_core.parse_pairs"""
    out = []
    for item in text.replace(" ", "").split(","):
        if not item: continue
        legs = item.upper().split("/")
        if len(legs) != 2 or not all(legs) or legs[0] == legs[1]: raise ValueError(f"bad pair '{item}'")
        if tuple(legs) in out: raise ValueError(f"duplicate pair '{item}'")
        out.append(tuple(legs))
    return out

PAIRS = parse_pairs(PAIRS_STR)

# Pull in configs
try:
    from screen_config import CONFIGS, TUNING
except ImportError as ex:
    raise ImportError("screen_config.py must be in the same QuantConnect project as this notebook") from ex
C = CONFIGS[CONFIG_VER]
assert C["cadf"] in ("both", "min") and C["correction"] in ("none", "bonferroni"), "unknown cadf / correction in screen_config.py"
FORMATION_YEARS  = C["formation_years"]
ALPHA            = C["alpha"]
MIN_PRICE        = C["min_price"]
MIN_DOLLAR_VOL   = C["min_dollar_vol"]
HL_RANGE         = C["hl_range"]
MAX_BETA_DRIFT   = C["max_beta_drift"]
MIN_CROSS_PER_YR = C["min_cross_per_yr"]
MAX_START_GAP    = C["max_start_gap"]
F_START, F_END = AS_OF - timedelta(days=round(365.25 * FORMATION_YEARS)), AS_OF
print(f"config {CONFIG_VER}: {C}")

In [ ]:
def screen_pair(ca, cb, va, vb, n_trials):
    """Formation-window screen for one pair. ca/cb = adjusted closes, va/vb = volumes. Returns stats + pass/fail + reasons."""
    out, why = {}, []

    # Listed for the whole window (point-in-time: a name that didn't exist yet fails here)
    for nm, c in (("A", ca), ("B", cb)):
        c = c.dropna()
        if c.empty or (c.index[0] - pd.Timestamp(F_START)).days > MAX_START_GAP:
            why.append(f"{nm} not listed full window")
    if why:
        return dict(passed=False, reasons="; ".join(why))

    # Liquidity floor
    for nm, c, v in (("A", ca, va), ("B", cb, vb)):
        out[f"min_px_{nm}"] = round(float(c.min()), 2)
        out[f"med_dv_{nm}"] = round(float((c * v).median()), 0)
        if c.min() <= MIN_PRICE: why.append(f"{nm} price < {MIN_PRICE}")
        if (c * v).median() < MIN_DOLLAR_VOL: why.append(f"{nm} dollar vol < {MIN_DOLLAR_VOL:,.0f}")

    df = pd.concat([np.log(ca), np.log(cb)], axis=1).dropna(); df.columns = ["a", "b"]
    years = len(df) / 252

    # Engle-Granger (CADF) both directions; config cadf="both" needs both below the cutoff (larger p), "min" either (smaller p)
    p_ab, p_ba = coint(df.a, df.b)[1], coint(df.b, df.a)[1]
    p = max(p_ab, p_ba) if C["cadf"] == "both" else min(p_ab, p_ba)
    alpha_adj = ALPHA / max(n_trials, 1) if C["correction"] == "bonferroni" else ALPHA   # "none" = plain alpha per pair
    out.update(p_ab=round(p_ab, 4), p_ba=round(p_ba, 4), alpha_adj=round(alpha_adj, 5))
    if p >= alpha_adj: why.append("CADF fails")

    # Hedge ratio (OLS) and its stability across the two halves of the window
    beta = np.polyfit(df.b, df.a, 1)[0]
    h = len(df) // 2
    b1, b2 = np.polyfit(df.b[:h], df.a[:h], 1)[0], np.polyfit(df.b[h:], df.a[h:], 1)[0]
    drift = abs(b1 - b2) / abs(beta) if beta != 0 else np.inf
    out.update(beta=round(float(beta), 3), beta_h1=round(float(b1), 3), beta_h2=round(float(b2), 3), beta_drift=round(float(drift), 2))
    if min(beta, b1, b2) <= 0: why.append("beta <= 0 in a half")
    if drift > MAX_BETA_DRIFT: why.append("beta unstable")

    # Half-life from AR(1): ds = lam * s(t-1) + e  (same formula as pairs_core)
    s = df.a - beta * df.b
    lam = np.polyfit(s.shift(1).dropna(), s.diff().dropna(), 1)[0]
    hl = -np.log(2) / lam if lam < 0 else np.inf
    out["half_life"] = round(float(hl), 1)
    if not (HL_RANGE[0] <= hl <= HL_RANGE[1]): why.append("half-life out of range")

    # Zero-crossings per year (spread demeaned over the window), so one episode can't drive the result
    sgn = np.sign(s - s.mean())
    cross = float((sgn.diff().abs() > 0).sum()) / years
    out["cross_per_yr"] = round(cross, 1)
    if cross < MIN_CROSS_PER_YR: why.append("too few crossings")

    out.update(passed=not why, reasons="; ".join(why) or "pass")
    return out

In [ ]:
qb = QuantBook()
SYMBOLS = {}   # ticker -> Symbol at AS_OF; filled by the pool builder in cell 4

def fetch(pairs):
    """Adjusted daily closes and volumes for every ticker in pairs, formation window only."""
    tick = sorted({t for p in pairs for t in p})
    # Use the point-in-time Symbol from the pool when we have it (survives later renames/delistings);
    # otherwise add by ticker (batch mode: skill tickers are current names)
    syms = {t: SYMBOLS[t] if t in SYMBOLS else qb.add_equity(t, Resolution.DAILY).symbol for t in tick}
    h = qb.history(list(syms.values()), F_START, F_END, Resolution.DAILY)
    return syms, h["close"].unstack(level=0), h["volume"].unstack(level=0)

def select_top(rows):
    """Tuning: pairs whose only failure is the CADF cutoff are eligible; rank them by CADF p-value
    (same direction rule as the screen) and mark the best TUNING['n_select'] as passed."""
    def eligible(r):
        return set(str(r["reasons"]).split("; ")) <= {"pass", "CADF fails"}
    pick = max if C["cadf"] == "both" else min        # same p the screen uses
    ok = sorted((r for r in rows if eligible(r)), key=lambda r: pick(r["p_ab"], r["p_ba"]))
    for k, r in enumerate(ok, 1):
        r["rank"] = k
        if k <= TUNING["n_select"]:
            r.update(passed=True, reasons=f"selected (rank {k} of {len(ok)} eligible)")
        else:
            r.update(passed=False, reasons=f"eligible, not in top {TUNING['n_select']} (rank {k})")
    return rows

def screen_group(pairs, group, rule="cadf"):
    """Screens one group of pairs. rule="cadf": Bonferroni cutoff per group (batch). rule="rank": top-N by p-value (tuning)."""
    syms, C, V = fetch(pairs)
    rows = []
    for a, b in pairs:
        sa, sb = syms[a], syms[b]
        if sa not in C.columns or sb not in C.columns:
            rows.append(dict(group=group, pair=f"{a}/{b}", passed=False, reasons="no data")); continue
        r = screen_pair(C[sa], C[sb], V[sa], V[sb], n_trials=len(pairs))
        rows.append(dict(group=group, pair=f"{a}/{b}", **r))
    return select_top(rows) if rule == "rank" else rows

In [ ]:
# Baselines: same screen, same trial count, so pass rates are comparable with the skill's batch

# Pool as of AS_OF from QuantConnect's Morningstar data (point-in-time, no hand-picking)
TOP_N = C["pool_top_n"]              # Most liquid stocks at AS_OF; must match selection.md

def _ok(x):
    code = x.asset_classification.morningstar_industry_code if x.has_fundamental_data else 0
    return x.has_fundamental_data and code and x.price > MIN_PRICE and x.dollar_volume > MIN_DOLLAR_VOL

def _liquid(fundamentals):
    return [x.symbol for x in sorted(filter(_ok, fundamentals), key=lambda x: x.dollar_volume, reverse=True)[:TOP_N]]

_u = qb.add_universe(_liquid)
_hist = qb.universe_history(_u, AS_OF - timedelta(days=7), AS_OF).droplevel("symbol", axis=0)
_hist = _hist[_hist.apply(len) > 0]                          # skip non-trading days
_pool = sorted(filter(_ok, _hist.iloc[-1]), key=lambda x: x.dollar_volume, reverse=True)[:TOP_N]   # selection closest to AS_OF
INDUSTRY = {x.symbol.value: int(x.asset_classification.morningstar_industry_code) for x in _pool}
SYMBOLS.update({x.symbol.value: x.symbol for x in _pool})
BASELINE_POOL = list(INDUSTRY)                               # random baseline draws from the same pool
print(f"Pool: {len(INDUSTRY)} stocks, {len(set(INDUSTRY.values()))} industries, as of {_hist.index[-1]:%Y-%m-%d}")
SEED = 0                  # Record the seed in the registry
N_BASE = len(PAIRS)

def random_pairs(pool, n, seed):
    rng = random.Random(seed); seen = set(); out = []
    while len(out) < n and len(pool) >= 2:
        a, b = rng.sample(pool, 2); k = tuple(sorted((a, b)))
        if k not in seen: seen.add(k); out.append((a, b))
    return out

def same_industry_pairs(industry, n, seed):
    """All same-industry pairs (n=None), or a random sample of n of them."""
    groups = {}
    for t, code in sorted(industry.items()): groups.setdefault(code, []).append(t)
    cands = [(a, b) for g in groups.values() for i, a in enumerate(g) for b in g[i + 1:]]
    return cands if n is None or n >= len(cands) else random.Random(seed).sample(cands, n)

In [ ]:
if MODE == "tuning":
    # Rule-based tuning set: same-industry pairs as of AS_OF, screened with the same frozen thresholds
    assert INDUSTRY, "pool is empty: check cell 4"
    MAIN_GROUP = "tuning"
    cands = same_industry_pairs(INDUSTRY, TUNING["max_candidates"], TUNING["seed"])
    print(f"Screening {len(cands)} same-industry candidates")
    rows = screen_group(cands, MAIN_GROUP, rule="rank")
else:
    # Skill batch plus baselines with the same trial count
    assert PAIRS, "paste the skill's Pairs (paste) line into PAIRS_STR"
    MAIN_GROUP = f"skill_{SKILL_VER}"
    rows = screen_group(PAIRS, MAIN_GROUP)
    if BASELINE_POOL:
        rows += screen_group(random_pairs(BASELINE_POOL, N_BASE, SEED), "random")
    if INDUSTRY:
        rows += screen_group(same_industry_pairs(INDUSTRY, N_BASE, SEED), "same_industry")

res = pd.DataFrame(rows)
summary = res.groupby("group").agg(n=("pair", "size"), passed=("passed", "sum"))
summary["pass_rate"] = (summary.passed / summary.n).round(2)
if "rank" in res:
    summary["eligible"] = res.groupby("group")["rank"].count()
print(f"{MODE} | batch {BATCH_ID} | formation {F_START:%Y-%m-%d} to {F_END:%Y-%m-%d}")
print(summary)
res

In [ ]:
# Paste-ready output for the repo: copy each block between its BEGIN/END lines
# (or select all output, paste into a .txt, and run tools/extract_blocks.py on it)
def repo_block(name, text):
    print(f"===== BEGIN {name} =====\n{text.rstrip()}\n===== END {name} =====\n")

# batches/<BATCH_ID>/screen.csv
repo_block("screen.csv", res.to_csv(index=False))

# Lines to append to pair_registry.md under this batch
rates = " | ".join(f"{g} {int(r.passed)}/{int(r.n)}" for g, r in summary.iterrows())
passed = res[(res.group == MAIN_GROUP) & (res.passed == True)].pair.tolist()
repo_block("pair_registry.md", "\n".join([
    f"### {BATCH_ID} screen ({datetime.now():%Y-%m-%d}, {MODE} mode)",
    f"- config {CONFIG_VER} | code {{CODE_COMMIT}} | T={AS_OF:%Y-%m-%d} | formation {F_START:%Y-%m-%d} to {F_END:%Y-%m-%d} | seed {SEED}",
    (f"- Rule: rank by CADF p (both directions), top {TUNING['n_select']} of {int(summary.loc[MAIN_GROUP].get('eligible', 0))} eligible from {len(res)} candidates"
     if MODE == "tuning" else f"- Rule: CADF ({C['cadf']} directions), {C['correction']} correction"),
    f"- Passed: {rates}",
    f"- To backtest ({len(passed)}): paste into the QC `pairs` parameter, and set batch_id={BATCH_ID}, start={AS_OF + timedelta(days=1):%Y-%m-%d}",
    f"- pairs: {','.join(passed) or 'none'}",
]))

# Pair diagnostics
Run before the backtest. Use the same `PAIRS` list as `main.py`.

Rough guide: `coint_p` < 0.05, `half_life_days` between about 3 and 40, a narrow `beta_min`–`beta_max` range, and a high `share_of_years_coint` are good signs. These are full-sample statistics, so treat them as a screen, not proof.

Note: this section uses the full 2014–2025 sample, so it is for diagnosing pairs after the fact, never for selecting them.

In [ ]:
from datetime import datetime
# PLACEHOLDER: pairs to diagnose, e.g. the screen's "- pairs:" line. Reuses the name PAIRS, so run this section after the screen
DIAG_STR = ""
PAIRS = [tuple(p.split("/")) for p in DIAG_STR.replace(" ", "").upper().split(",") if p]
START, END = datetime(2014, 1, 1), datetime(2025, 12, 31)

qb = QuantBook()
syms = {t: qb.add_equity(t, Resolution.DAILY).symbol for p in PAIRS for t in p}
closes = qb.history(list(syms.values()), START, END, Resolution.DAILY)["close"].unstack(level=0)

In [ ]:
import numpy as np, pandas as pd
from statsmodels.tsa.stattools import coint

def diagnose(a: pd.Series, b: pd.Series, window=252):
    """Statistical health of one pair from two aligned close-price series."""
    df = pd.concat([np.log(a), np.log(b)], axis=1).dropna(); df.columns = ["a", "b"]
    beta = np.polyfit(df.b, df.a, 1)[0]
    s = df.a - beta * df.b
    lam = np.polyfit(s.shift(1).dropna(), s.diff().dropna(), 1)[0]      # ds = lam * s(t-1) + e
    roll_beta = df.a.rolling(window).cov(df.b) / df.b.rolling(window).var()
    yearly_p = {int(y): round(coint(g.a, g.b)[1], 3) for y, g in df.groupby(df.index.year) if len(g) > 120}
    return dict(coint_p=round(coint(df.a, df.b)[1], 4), beta=round(beta, 3),
                half_life_days=round(-np.log(2) / lam, 1) if lam < 0 else float("inf"),
                beta_min=round(roll_beta.min(), 2), beta_max=round(roll_beta.max(), 2),
                return_corr=round(df.a.diff().corr(df.b.diff()), 2),
                share_of_years_coint=round(np.mean([p < 0.10 for p in yearly_p.values()]), 2),
                coint_p_by_year=yearly_p)

rows = {f"{a}/{b}": diagnose(closes[syms[a]], closes[syms[b]]) for a, b in PAIRS}
by_year = pd.DataFrame({k: v.pop("coint_p_by_year") for k, v in rows.items()})
pd.DataFrame(rows).T

In [ ]:
by_year  # cointegration p-value within each calendar year

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(PAIRS), 1, figsize=(11, 3 * len(PAIRS)), squeeze=False)
for ax, (a, b) in zip(axes[:, 0], PAIRS):
    la, lb = np.log(closes[syms[a]]), np.log(closes[syms[b]])
    s = la - rows[f"{a}/{b}"]["beta"] * lb
    ((s - s.rolling(60).mean()) / s.rolling(60).std()).plot(ax=ax, lw=0.8)
    for y in (-2, 0, 2): ax.axhline(y, color="grey", lw=0.6, ls="--")
    ax.set_title(f"{a}/{b} 60-day z-score of spread")
plt.tight_layout()